In [4]:
%load_ext autoreload
%autoreload 2

import xarray as xr
import torch
import yaml
import sys
from pathlib import Path
root_dir = Path.cwd().parent   
sys.path.append(str(root_dir))

from data.dataset import ERA5Dataset
from data.dataloader import *
from data.preprocessing import *

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (runtime):", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Torch version: 2.3.1
CUDA available: True
CUDA version (runtime): 12.1
GPU: Quadro T1000 with Max-Q Design


Load the configurations and the full dataset. Then, reduce the dataset for the intended size and split by train/val/test. 

In [ ]:
# Load config
config_path = Path.cwd().parent / "utils" / "default_config.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)
print("Config loaded!")

# Load full dataset
url = config["data"]["dataset_url"]
ds = load_full_dataset(url)

# Reduce dataset using config
splits = reduce_dataset(ds, config)
train_ds = splits["train"]
val_ds   = splits["val"]
test_ds  = splits["test"]

Config loaded!
Opening dataset from: gs://weatherbench2/datasets/era5_daily/1959-2023_01_10-full_37-1h-0p25deg-chunk-1-s2s.zarr
Full dataset loaded!
Reducing dataset from source (this may take a while)...
Creating train set (2017-01-01 → 2018-12-31)...


Get the dataloaders for each split, ready to plug into the ML pipeline.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

variables_to_keep = config["data"]["variables_to_keep"]
target_var = config["data"]["target_variable"]
input_length = config["training"]["input_length"]
forecast_horizon = config["training"]["forecast_horizon"]
batch_size = config["training"]["batch_size"]
num_workers = config["training"]["num_workers"]

train_loader, val_loader, test_loader = get_dataloaders(
    input_vars=variables_to_keep, 
    target_var=target_var,
    config=config,
    input_length=input_length, 
    forecast_horizon=forecast_horizon,
    batch_size=batch_size, 
    num_workers=0,
    load_into_ram=True,
    device=device
)
print("\nDataloaders ready!")

In [ ]:
X, y = next(iter(train_loader)) 
print("Device:", X.device, y.device)